# Denormalised View for BI Reporting

This notebook creates a denormalised view in the **Gold** layer by joining the fact order items table with date and product dimension tables. The resulting view consolidates transactional data with dimensional attributes and a derived time-of-day field, making it ready for BI dashboard consumption without requiring downstream joins.

## Source Tables

| Table | Layer | Role | Key Columns |
| --- | --- | --- | --- |
| `ecommerce.gold.gld_fact_order_items` | Gold | Fact | `date_id`, `product_id`, `transaction_ts` |
| `ecommerce.gold.gld_dim_date` | Gold | Dimension | `date_id` |
| `ecommerce.gold.gld_dim_products` | Gold | Dimension | `product_id` |

### Join Strategy

* **Fact → Date**: `LEFT JOIN` on `date_id` — preserves all transactions even if a date record is missing
* **Fact → Product**: `LEFT JOIN` on `product_id` — preserves all transactions even if a product record is missing

A derived column `hour_of_day` is extracted from `transaction_ts` to support time-of-day analysis in dashboards.

In [0]:
%sql

-- ============================================================
-- Create a denormalised view for BI Dashboard reporting
-- ============================================================
-- This view joins the fact order items table with date and
-- product dimension tables to produce a wide, analysis-ready
-- table. LEFT JOINs preserve all fact rows even when a
-- matching dimension record is missing.
--
-- Source tables:
--   o → ecommerce.gold.gld_fact_order_items (fact)
--   d → ecommerce.gold.gld_dim_date         (date dimension)
--   p → ecommerce.gold.gld_dim_products     (product dimension)
-- ============================================================

CREATE OR REPLACE VIEW ecommerce.gold.fact_transactions_denorm AS (
    SELECT
        -- All columns from the fact table (transactional data)
        o.*,
        -- Date dimension attributes
        d.year,
        d.month_name,
        d.day_name,
        d.is_weekend,
        d.quarter,
        d.quarter_number,
        d.week_of_year,
        d.week_number,
        -- Product dimension attributes
        p.sku, 
        p.category_code,
        p.category_name,
        p.brand_code,
        p.brand_name,
        p.color,
        p.size,
        p.rating_count,
        -- Derived: extract the hour from the transaction timestamp
extract(HOUR FROM transaction_ts) as hour_of_day
    FROM ecommerce.gold.gld_fact_order_items o
    LEFT JOIN ecommerce.gold.gld_dim_date d ON o.date_id = d.date_id
    LEFT JOIN ecommerce.gold.gld_dim_products p ON p.product_id = o.product_id 
);

## Validation

After creating the view, we validate it by checking the total row count and previewing sample rows to ensure the joins and column selections are correct.

In [0]:
%sql

-- Verify total row count in the denormalised view
SELECT COUNT(*) AS total_rows FROM ecommerce.gold.fact_transactions_denorm;

In [0]:
%sql

-- Preview the first 100 rows to inspect the denormalised data
SELECT * FROM ecommerce.gold.fact_transactions_denorm LIMIT 100;

## Summary

The `ecommerce.gold.fact_transactions_denorm` view is now available for BI dashboards. It combines:

* **Transactional data** — all columns from the fact table (`o.*`)
* **Date attributes** — year, month name, day name, weekend flag, quarter, quarter number, week of year, week number
* **Product attributes** — SKU, category code, category name, brand code, brand name, color, size, rating count
* **Derived time attribute** — `hour_of_day` extracted from `transaction_ts`

This wide format eliminates the need for dashboard consumers to perform joins, simplifying report creation and ensuring consistent data access across the organisation.